# Project 5 — Iterative Methods for Linear Systems
**Topic:** Solving $\mathbf{A}\mathbf{u} = \mathbf{b}$ iteratively (Jacobi, Gauss-Seidel, SOR) applied to the 1-D Poisson equation. Convergence analysis and spectral radius.


---
## 1. Set Up

### Physical Motivation
The 1-D **Poisson equation** $-u''(x) = f(x)$ arises in:
- **Electrostatics**: $-\nabla^2 \phi = \rho/\varepsilon_0$ (electrostatic potential from charge density).
- **Heat conduction**: $-\kappa u'' = q$ (steady-state temperature with heat source $q$).
- **Quantum mechanics**: the discretised Hamiltonian for a particle in a potential.

After finite differencing on a uniform grid with spacing $h = 1/(n+1)$:
$$-\frac{u_{i-1} - 2u_i + u_{i+1}}{h^2} = f_i$$

This gives a tridiagonal linear system $\mathbf{A}\mathbf{u} = \mathbf{b}$ where $\mathbf{A}$ has the **standard 1-D Laplacian structure**: diagonal $+2/h^2$, off-diagonals $-1/h^2$.

### Why Iterative Methods?
For a 1-D problem, a tridiagonal solver ($\mathcal{O}(n)$) is optimal. But for **2-D or 3-D** problems, the matrix becomes block-tridiagonal with bandwidth $\sqrt{n}$ — direct methods cost $\mathcal{O}(n^{3/2})$ in 2-D. Iterative methods scale better and are essential for large sparse systems.

### Convergence Theory
All three methods are **stationary iterations** $\mathbf{u}^{k+1} = \mathbf{M}\mathbf{u}^k + \mathbf{c}$. The error $\mathbf{e}^k = \mathbf{u}^k - \mathbf{u}^*$ evolves as $\mathbf{e}^k = \mathbf{M}^k \mathbf{e}^0$. Convergence requires the **spectral radius** $\rho(\mathbf{M}) < 1$.

For the Poisson equation on $[0,1]$:
- **Jacobi**: $\rho_J = \cos(\pi h)$ — convergence rate per step is $\mathcal{O}(1 - \pi^2 h^2/2)$.
- **Gauss-Seidel**: $\rho_{GS} = \rho_J^2$ — **twice as fast** as Jacobi per iteration.
- **SOR (optimal $\omega$)**: $\rho_{\text{SOR}} = \omega_{\text{opt}} - 1$ where $\omega_{\text{opt}} = 2/(1 + \sin(\pi h))$ — dramatically faster.

### ⚠️ Limitations
1. **Slow convergence for large $n$**: Jacobi requires $\mathcal{O}(n^2)$ iterations to converge — each doubling of $n$ quadruples the work.
2. **No preconditioning**: for ill-conditioned systems (e.g. $h \ll 1$), convergence can be arbitrarily slow without a preconditioner.
3. **Python inner loops**: the Gauss-Seidel implementation uses a Python `for` loop over grid points — orders of magnitude slower than a C implementation.


In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("output_iterative_linear_systems")
OUT.mkdir(exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print(f"Output directory: {OUT.resolve()}")


---
## 2. Functions

### `poisson_1d_grid(n)`
Sets up $-u'' = \pi^2 \sin(\pi x)$ — the **manufactured solution** $u^*(x) = \sin(\pi x)$, which satisfies the Dirichlet BCs $u(0) = u(1) = 0$ exactly. This allows direct error measurement against the true solution.

```python
b = (np.pi**2) * np.sin(np.pi * x)
```
The right-hand side: if $u^* = \sin(\pi x)$, then $-u^{*''} = \pi^2\sin(\pi x)$. Using a manufactured solution with a known closed form is the standard way to **verify** a numerical solver.

### `residual_poisson_1d(u, b, h)` — The L∞ Residual
```python
Au[1:-1] = (2*u[1:-1] - u[:-2] - u[2:]) / h**2
return np.linalg.norm(Au - b, ord=np.inf)
```
Computes $\|\mathbf{A}\mathbf{u} - \mathbf{b}\|_\infty$ — the **maximum pointwise residual**. This measures how well the current iterate satisfies the linear system. **Not the same as the error** $\|u - u^*\|$ unless $\mathbf{A}$ is well-conditioned.

### `jacobi_poisson(b, h, ...)` — Jacobi Iteration
Update rule: each component updated using **only old values**:
$$u_i^{k+1} = \frac{h^2 b_i + u_{i-1}^k + u_{i+1}^k}{2}$$

```python
u_new = u.copy()
u_new[1:-1] = 0.5 * (b[1:-1] * h**2 + u[:-2] + u[2:])
u = u_new
```
The `.copy()` is **essential** — it ensures all updates use the old values. Omitting `.copy()` accidentally implements Gauss-Seidel. This is a common bug.

### `gauss_seidel_poisson(b, h, ..., omega)` — SOR
```python
candidate = 0.5 * (b[i] * h**2 + left + right)
u[i] = (1.0 - omega) * u[i] + omega * candidate
```
**Key**: uses the **just-updated** $u_{i-1}$ (left neighbour already updated in this sweep). This is what makes Gauss-Seidel faster than Jacobi — new information propagates immediately.

The `omega` parameter: $\omega = 1$ is pure Gauss-Seidel; $\omega > 1$ is **over-relaxation** (SOR); $0 < \omega < 1$ is under-relaxation. The optimal $\omega_{\text{opt}} = 2/(1 + \sin(\pi h))$ minimises the spectral radius.

### 🔧 Improvement Directions
1. **Multigrid**: solves the Poisson equation in $\mathcal{O}(n)$ operations — optimal for any $n$. The key idea: smooth low-frequency errors on a coarse grid, high-frequency errors on the fine grid.
2. **Conjugate Gradient**: for symmetric positive-definite $\mathbf{A}$, CG converges in $\mathcal{O}(\sqrt{\kappa})$ iterations where $\kappa$ is the condition number.
3. **Vectorised Gauss-Seidel**: use red-black ordering (checkerboard pattern) to vectorise the inner loop while preserving the sequential update advantage.

### ⚠️ Known Weaknesses
- **Python loop in Gauss-Seidel**: $\mathcal{O}(n)$ Python iterations per sweep is very slow. In production code, use SciPy sparse solvers.
- **Residual vs. error**: convergence in residual does NOT guarantee convergence in error if $\mathbf{A}$ is ill-conditioned. Always verify against the analytic solution when available.


In [ ]:
def poisson_1d_grid(n):
    h = 1.0 / (n + 1)
    x = np.linspace(h, 1.0 - h, n)
    b = (np.pi**2) * np.sin(np.pi * x)    # manufactured RHS: -u''=pi^2*sin(pi*x)
    return x, h, b                         # exact solution: u*(x)=sin(pi*x)

def residual_poisson_1d(u, b, h):
    Au = np.empty_like(u)
    Au[0]    = (2*u[0]  - u[1]) / h**2
    Au[-1]   = (2*u[-1] - u[-2]) / h**2
    Au[1:-1] = (2*u[1:-1] - u[:-2] - u[2:]) / h**2
    return np.linalg.norm(Au - b, ord=np.inf)

def jacobi_poisson(b, h, tol=1e-8, max_iter=8000):
    u = np.zeros_like(b)
    residuals = []
    for it in range(max_iter):
        u_new = u.copy()                           # MUST copy — Jacobi uses all old values
        u_new[0]    = 0.5*(b[0]   *h**2 + u[1])
        u_new[-1]   = 0.5*(b[-1]  *h**2 + u[-2])
        u_new[1:-1] = 0.5*(b[1:-1]*h**2 + u[:-2] + u[2:])
        u = u_new
        res = residual_poisson_1d(u, b, h)
        residuals.append(res)
        if res < tol: break
    return u, np.array(residuals)

def gauss_seidel_poisson(b, h, tol=1e-8, max_iter=8000, omega=1.0):
    u = np.zeros_like(b)
    residuals = []
    n = len(b)
    for it in range(max_iter):
        for i in range(n):
            left  = u[i-1] if i > 0   else 0.0    # already updated (new value)
            right = u[i+1] if i < n-1 else 0.0    # not yet updated (old value)
            cand  = 0.5*(b[i]*h**2 + left + right)
            u[i]  = (1.0 - omega)*u[i] + omega*cand   # SOR blending
        res = residual_poisson_1d(u, b, h)
        residuals.append(res)
        if res < tol: break
    return u, np.array(residuals)


---
## 3 & 4. Calculation & Writeouts

Compare iteration counts and convergence rates. SOR with optimal $\omega$ should converge in $\mathcal{O}(\sqrt{n})$ iterations vs. $\mathcal{O}(n^2)$ for Jacobi.

The optimal $\omega$:
$$\omega_{\text{opt}} = \frac{2}{1 + \sin(\pi h)}$$

For $n=50$: $h = 1/51 \approx 0.02$, $\omega_{\text{opt}} \approx 2/(1+\sin(0.06)) \approx 1.88$.


In [ ]:
n = 50
x, h, b = poisson_1d_grid(n)
u_exact = np.sin(np.pi * x)

omega_opt = 2.0 / (1.0 + np.sin(np.pi * h))
print(f"Grid: n={n}, h={h:.4f}")
print(f"Optimal SOR omega: {omega_opt:.5f}")
print()

u_jac, res_jac  = jacobi_poisson(b, h)
u_gs,  res_gs   = gauss_seidel_poisson(b, h, omega=1.0)
u_sor, res_sor  = gauss_seidel_poisson(b, h, omega=omega_opt)

for label, u, res in [("Jacobi",    u_jac, res_jac),
                       ("Gauss-Seidel", u_gs, res_gs),
                       ("SOR",       u_sor, res_sor)]:
    err = np.max(np.abs(u - u_exact))
    print(f"{label:15s}: {len(res):5d} iters, "
          f"residual={res[-1]:.2e}, max_error={err:.2e}")


---
## 5. Matplotlib & 6. Sanity Checks

### Interpreting Convergence Plots
- **Semilog**: residual vs. iterations. Straight lines on semilog = **geometric (linear) convergence**. Slope gives the log of the spectral radius: $\log_{10}(\rho)$.
- **SOR vs. GS**: SOR curve should be much steeper (faster convergence) — the entire point of the method.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Convergence curves
ax = axes[0]
for label, res, c in [("Jacobi", res_jac, 'steelblue'),
                       ("Gauss-Seidel", res_gs, 'tomato'),
                       (f"SOR ω={omega_opt:.3f}", res_sor, 'darkgreen')]:
    ax.semilogy(res, color=c, lw=2, label=label)
ax.set_xlabel('Iteration'); ax.set_ylabel('$\|Au-b\|_\infty$')
ax.set_title('Convergence comparison')
ax.legend(); ax.grid(True, which='both', alpha=0.4)

# Solution accuracy
ax2 = axes[1]
ax2.plot(x, u_exact, 'k--', lw=2, label='Exact $\sin(\pi x)$')
ax2.plot(x, u_sor, 'darkgreen', lw=1.5, label='SOR solution')
ax2.plot(x, np.abs(u_sor - u_exact)*100, 'r:', lw=1.2, label='Error × 100')
ax2.set_xlabel('$x$'); ax2.set_ylabel('$u(x)$')
ax2.set_title('Solution vs. exact')
ax2.legend()

plt.tight_layout()
plt.savefig(OUT / "iterative_linear.png", dpi=150, bbox_inches='tight')
plt.show()

# Sanity checks
print("=" * 55)
print("SANITY CHECKS — Project 05 Iterative Linear Systems")
print("=" * 55)

u_methods = [("Jacobi", u_jac), ("Gauss-Seidel", u_gs), ("SOR", u_sor)]
for name, u in u_methods:
    err     = np.max(np.abs(u - u_exact))
    res     = residual_poisson_1d(u, b, h)
    ok_err  = err < 1e-4
    ok_res  = res < 1e-8
    ok_bc   = u[0] > 0 and u[-1] > 0   # solution should be positive interior
    print(f"  {name:15s}: max_err={err:.2e} {'✓' if ok_err else '✗'}  "
          f"final_res={res:.2e} {'✓' if ok_res else '✗'}")

# SOR should be fastest
ok_sor = len(res_sor) < len(res_jac) and len(res_sor) < len(res_gs)
print(f"\n  SOR converges fastest: {'✓' if ok_sor else '✗'} "
      f"(Jac:{len(res_jac)}, GS:{len(res_gs)}, SOR:{len(res_sor)} iters)")
print("=" * 55)
